# Task 02A — full SAAS structural-posterior reuse diagnostic

This notebook is the reproducible Google Colab driver for the full Task 02A experiment. It only runs fixed-particle SAAS reweighting and fresh-reference diagnostics; it does not implement Task 02B, particle movement, q>1 BO, or a BO loop. Colab currently lists Python 3.12.13 in its 2026.04 runtime, which is compatible with this project; the runtime guard below checks the actual interpreter rather than assuming it remains fixed.

Before running it:

1. Publish the Task 02A source to the GitHub revision selected below. Colab clones that revision; it cannot see unpushed local work.
2. For the reproducible default, leave the Colab runtime on CPU and keep `ACCELERATOR = "cpu"`. For optional NUTS acceleration, choose an NVIDIA GPU in **Runtime → Change runtime type** before running any code, then set `ACCELERATOR = "gpu"`.
3. Run the setup and preflight cells. The expensive full run is guarded by `RUN_FULL = False`; set it to `True` only after preflight succeeds.

The exact GP cache and diagnostics intentionally remain on CPU. An NVIDIA GPU only accelerates the JAX/NumPyro SAAS NUTS reference fits. The final cell downloads one ZIP for you to review and publish manually. This notebook never authenticates to, commits to, or pushes to GitHub.


In [ ]:
# User configuration. A commit SHA is preferable to main for a final scientific run.
from pathlib import Path
import os

REPOSITORY_URL = "https://github.com/PaulsonLab/energy-inference-bo.git"
REPO_REF = "main"  # Replace with the published Task 02A commit SHA for a pinned run.
ACCELERATOR = "cpu"  # Allowed values: "cpu" (default) or "gpu".
RUN_FULL = False  # Safety guard: set True only after the preflight cell passes.

if ACCELERATOR not in {"cpu", "gpu"}:
    raise ValueError("ACCELERATOR must be 'cpu' or 'gpu'")

# Set before importing JAX. GPU mode avoids JAX reserving most GPU memory.
os.environ["JAX_PLATFORMS"] = "cuda" if ACCELERATOR == "gpu" else "cpu"
if ACCELERATOR == "gpu":
    os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

REPO_DIR = Path("/content/energy-inference-bo")
OUTPUT_DIR = REPO_DIR / "artifacts/task02a_full"
SUMMARY_PATH = REPO_DIR / "TASK_02A_COLAB_SUMMARY.md"
MANIFEST_PATH = REPO_DIR / "colab_manifest.json"


In [ ]:
# Runtime guard: the project supports Python 3.11 and 3.12 only.
import platform
import subprocess
import sys

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError(
        "This repository requires Python 3.11 or 3.12. Choose a compatible Colab runtime version."
    )

if ACCELERATOR == "gpu":
    gpu_check = subprocess.run(["nvidia-smi"], text=True, capture_output=True)
    if gpu_check.returncode != 0:
        raise RuntimeError(
            "GPU mode was selected but nvidia-smi is unavailable. Select an NVIDIA GPU runtime or use CPU mode."
        )
    print(gpu_check.stdout)
else:
    print("CPU mode selected; this is the reproducible default.")


In [ ]:
# Clone exactly the requested GitHub revision without using any credentials.
def run(command, *, cwd=None):
    print("+", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, check=True)

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git clone; use a fresh Colab runtime.")
    run(["git", "fetch", "--tags", "origin"], cwd=REPO_DIR)
else:
    run(["git", "clone", REPOSITORY_URL, str(REPO_DIR)])

run(["git", "checkout", "--detach", REPO_REF], cwd=REPO_DIR)
GIT_SHA = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
print(f"Checked out: {GIT_SHA}")
os.chdir(REPO_DIR)


In [ ]:
# Install only locked project dependencies. Do not import JAX before this cell completes.
run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=REPO_DIR)

if ACCELERATOR == "gpu":
    jax_version = next(
        line.split("==", 1)[1].split(";", 1)[0].strip()
        for line in (REPO_DIR / "requirements.txt").read_text().splitlines()
        if line.startswith("jax==")
    )
    # The CUDA plugin version is pinned to the repository's locked JAX version.
    run(
        [sys.executable, "-m", "pip", "install", f"jax[cuda12]=={jax_version}"],
        cwd=REPO_DIR,
    )

run([sys.executable, "-m", "pip", "install", "-e", "."], cwd=REPO_DIR)


In [ ]:
# Preflight: prove the chosen JAX backend and run the complete test suite.
import importlib.metadata

import jax

PACKAGE_NAMES = ("torch", "botorch", "gpytorch", "jax", "jaxlib", "numpyro", "numpy", "scipy")
PACKAGE_VERSIONS = {name: importlib.metadata.version(name) for name in PACKAGE_NAMES}
print(PACKAGE_VERSIONS)
print(f"JAX backend: {jax.default_backend()}")
print(f"JAX devices: {jax.devices()}")

if ACCELERATOR == "cpu" and jax.default_backend() != "cpu":
    raise RuntimeError("CPU mode requested, but JAX did not select the CPU backend.")
if ACCELERATOR == "gpu":
    if jax.default_backend() != "gpu" or not jax.devices():
        raise RuntimeError("GPU mode requested, but JAX did not initialize an NVIDIA GPU backend.")

run([sys.executable, "-m", "pytest", "-q"], cwd=REPO_DIR)
print("Preflight passed. The full run remains disabled until RUN_FULL is set to True.")


## Full experiment — explicit opt-in

Set `RUN_FULL = True` in the configuration cell, rerun it, then execute the next cell. The full configuration runs seeds 0, 1, and 2 at D=10, n=16→40 with fresh SAAS NUTS references. It can take hours and should not be launched accidentally by **Run all**.


In [ ]:
FULL_COMMAND = [
    sys.executable,
    "-m",
    "energy_bo.experiments.run_task02a",
    "--profile",
    "full",
    "--seeds",
    "0",
    "1",
    "2",
    "--output-dir",
    str(OUTPUT_DIR),
    "--summary-path",
    str(SUMMARY_PATH),
]

if RUN_FULL:
    run(FULL_COMMAND, cwd=REPO_DIR)
else:
    print("Full run skipped. Set RUN_FULL = True in the configuration cell after preflight succeeds.")


In [ ]:
# Save immutable run provenance after a successful full experiment.
import hashlib
import json
from datetime import datetime, timezone

if RUN_FULL:
    if not SUMMARY_PATH.is_file() or not OUTPUT_DIR.is_dir():
        raise RuntimeError("Expected Task 02A outputs are missing; do not create a partial archive.")
    manifest = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "repository_url": REPOSITORY_URL,
        "requested_ref": REPO_REF,
        "git_sha": GIT_SHA,
        "python": sys.version,
        "platform": platform.platform(),
        "accelerator": ACCELERATOR,
        "jax_backend": jax.default_backend(),
        "jax_devices": [str(device) for device in jax.devices()],
        "package_versions": PACKAGE_VERSIONS,
        "full_command": FULL_COMMAND,
        "requirements_sha256": hashlib.sha256((REPO_DIR / "requirements.txt").read_bytes()).hexdigest(),
    }
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + "\n")
    print(MANIFEST_PATH.read_text())
else:
    print("Manifest skipped because RUN_FULL is False.")


In [ ]:
# Download one reviewable archive. Nothing is uploaded to GitHub.
from zipfile import ZIP_DEFLATED, ZipFile

ARCHIVE_PATH = REPO_DIR / "task02a_full_outputs.zip"
if RUN_FULL:
    with ZipFile(ARCHIVE_PATH, "w", compression=ZIP_DEFLATED) as archive:
        archive.write(SUMMARY_PATH, SUMMARY_PATH.relative_to(REPO_DIR))
        archive.write(MANIFEST_PATH, MANIFEST_PATH.relative_to(REPO_DIR))
        for path in sorted(OUTPUT_DIR.rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(REPO_DIR))
    print(f"Created {ARCHIVE_PATH} ({ARCHIVE_PATH.stat().st_size:,} bytes)")
    from google.colab import files

    files.download(str(ARCHIVE_PATH))
else:
    print("Download skipped because RUN_FULL is False.")


## Manual publication after download

Review the ZIP locally. To publish results, copy `TASK_02A_COLAB_SUMMARY.md` into a tracked report in your local clone, optionally choose one figure or compact CSV to track, then use your normal local `git add`, `git commit`, and `git push` workflow. The detailed `artifacts/` directory is intentionally ignored and is not uploaded automatically.
